# Streamlining Workflows with Pipelines

When applying preprocessing techniques like standardization or PCA, it is necessary to reuse the parameters obtained from fitting the training data to correctly scale and compress any new data, such as a test dataset. To simplify this process, scikit-learn provides the `Pipeline` class, which allows you to fit a model consisting of an arbitrary number of transformation steps and apply it seamlessly to make predictions on new data.

<p align="center">
  <img src="https://raw.githubusercontent.com/rasbt/machine-learning-book/main/ch06/figures/06_01.png" width="600">
</p>



### Loading the Breast Cancer Wisconsin Dataset

In this example, we use the Breast Cancer Wisconsin dataset, which contains 569 examples of malignant and benign tumor cells. The dataset includes 30 real-valued features computed from digitized images of cell nuclei, alongside unique ID numbers and corresponding diagnoses ('M' for malignant, 'B' for benign).

In [1]:
import pandas as pd

# 1. Read in the dataset directly from the UCI website using pandas
df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data', header=None)

### Preprocessing and Splitting the Data

First, we assign the 30 features to a NumPy array `X` and the target labels to an array `y`. We then use a `LabelEncoder` to transform the original string representations of the classes ('M' and 'B') into integers (1 and 0, respectively). Finally, we divide the data into a training set (80%) and a separate test set (20%).

In [2]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 2. Assign features to X and labels to y, then encode labels
X = df.loc[:, 2:].values
y = df.loc[:, 1].values

le = LabelEncoder()
y = le.fit_transform(y)

# Double-check the mapping: 'M' -> 1, 'B' -> 0
print(le.classes_)
print(le.transform(['M', 'B']))

# 3. Divide the dataset into training and test splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=1
)

['B' 'M']
[1 0]


### Combining Transformers and Estimators in a Pipeline

Many learning algorithms require features to be on the same scale for optimal performance. To build our model, we will standardize the dataset columns, compress the data down to a two-dimensional subspace via Principal Component Analysis (PCA), and feed the transformed data to a Logistic Regression classifier.

Instead of performing these steps separately, we can streamline the process using the `make_pipeline` function:

* The `make_pipeline` function constructs a `Pipeline` object, which acts as a meta-estimator wrapping around individual scikit-learn transformers and estimators.
* When calling the `fit` method on a pipeline, the input data passes through a series of `fit` and `transform` calls on the intermediate steps until it arrives at the final estimator, which is then fitted to the fully transformed training data.
* Similarly, when the `predict` method is called, the data is routed through the intermediate `transform` steps before reaching the final estimator to generate predictions.
* A pipeline can have an unlimited number of intermediate steps, but the final element must be an estimator if you want to use the pipeline for prediction tasks.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# 4. Chain StandardScaler, PCA, and LogisticRegression objects in a pipeline
pipe_lr = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression()
)

# 5. Fit the entire pipeline to the training data
pipe_lr.fit(X_train, y_train)

# 6. Make predictions and evaluate accuracy on the test data
y_pred = pipe_lr.predict(X_test)
test_acc = pipe_lr.score(X_test, y_test)

print(f'Test accuracy: {test_acc:.3f}')

Test accuracy: 0.956
